<a href="https://colab.research.google.com/github/pozdnyavladimer-jpg/gitcube-os/blob/main/examples/experimental_loop.py" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Colab is alive")

Colab is alive


In [ ]:
import os

os.makedirs("core", exist_ok=True)
os.makedirs("runtime", exist_ok=True)

print("Folders created:", os.listdir())

Folders created: ['.config', 'core', 'runtime', 'sample_data']


In [ ]:
%%writefile core/state.py
from dataclasses import dataclass

STATE_KEYS = ["pressure", "flow", "structure", "balance", "law", "future"]


@dataclass
class SystemState:
    pressure: float
    flow: float
    structure: float
    balance: float
    law: float
    future: float

    def to_dict(self):
        return self.__dict__

    @classmethod
    def from_dict(cls, d):
        return cls(**d)


def normalize_state(d):
    d = {k: max(0.0, float(d.get(k, 0.0))) for k in STATE_KEYS}
    s = sum(d.values()) or 1.0
    return {k: d[k] / s for k in STATE_KEYS}


def default_state():
    return SystemState.from_dict(normalize_state({
        "pressure": 0.22,
        "flow": 0.14,
        "structure": 0.18,
        "balance": 0.14,
        "law": 0.14,
        "future": 0.18,
    }))

Writing core/state.py


In [ ]:
%%writefile core/evaluation.py
def compute_metrics(state):
    v = state.to_dict()

    shadow = v["pressure"]

    coherence = 1.0 - min(
        1.0,
        abs(v["balance"] - v["structure"]) +
        abs(v["balance"] - v["flow"])
    )

    target_fit = v["balance"] + v["structure"]
    vitality = v["flow"] + v["future"]

    return {
        "shadow": round(shadow, 3),
        "coherence": round(coherence, 3),
        "target_fit": round(target_fit, 3),
        "vitality": round(vitality, 3),
    }

Writing core/evaluation.py


In [ ]:
%%writefile runtime/agent_loop.py
import random
from core.state import SystemState, normalize_state
from core.evaluation import compute_metrics


CLASSES = {
    "TANK": {
        "pressure": -0.10,
        "structure": 0.24,
        "balance": 0.14,
        "law": 0.06,
    },
    "ARCHER": {
        "balance": 0.14,
        "structure": 0.08,
        "future": 0.20,
        "flow": 0.05,
    },
    "MAGE": {
        "flow": 0.22,
        "future": 0.22,
        "balance": 0.04,
        "pressure": 0.03,
    },
    "HEALER": {
        "balance": 0.28,
        "flow": 0.12,
        "pressure": -0.18,
        "law": 0.04,
    },
    "ASSASSIN": {
        "pressure": -0.05,
        "structure": -0.10,
        "future": 0.12,
        "balance": -0.05,
    },
}


def mutate(state, w):
    d = state.to_dict()
    new = {}

    for k, v in d.items():
        delta = w.get(k, 0.0) * random.uniform(0.65, 1.35)
        new[k] = v + delta

    return SystemState.from_dict(normalize_state(new))


def score(cls, m, v_current, history):
    s = (
        m["coherence"] * 1.45
        + m["vitality"] * 1.70
        + m["target_fit"] * 0.90
        - m["shadow"] * 1.10
    )

    recent = history[-10:]
    count_recent = recent.count(cls)
    if count_recent > 1:
        fatigue_map = {
            "TANK": 0.08,
            "ARCHER": 0.09,
            "MAGE": 0.11,
            "HEALER": 0.08,
            "ASSASSIN": 0.18,
        }
        s -= (count_recent - 1) * fatigue_map.get(cls, 0.1)

    if v_current < 0.30:
        if cls == "HEALER":
            s += 1.4
        elif cls == "TANK":
            s += 0.45
        elif cls == "MAGE":
            s -= 0.40
        elif cls == "ARCHER":
            s -= 0.80
        elif cls == "ASSASSIN":
            s -= 1.10

    elif v_current > 0.72:
        if cls == "MAGE":
            s += 1.0
        elif cls == "ARCHER":
            s += 0.35
        elif cls == "HEALER":
            s -= 0.10

    else:
        if cls == "ARCHER":
            s += 0.18
        if cls == "TANK":
            s += 0.12

    if cls == "TANK":
        if m["shadow"] > 0.12:
            s += 0.10
        if m["coherence"] < 0.86:
            s += 0.06

    elif cls == "HEALER":
        if m["shadow"] > 0.10:
            s += 0.16
        if m["coherence"] < 0.90:
            s += 0.08

    elif cls == "MAGE":
        if m["vitality"] > 0.72:
            s += 0.08
        if m["target_fit"] > 0.34:
            s += 0.06

    elif cls == "ARCHER":
        if m["target_fit"] > 0.34:
            s += 0.10
        if m["coherence"] > 0.88:
            s += 0.04

    elif cls == "ASSASSIN":
        s -= 0.18
        if m["shadow"] > 0.16:
            s += 0.22
        if m["coherence"] > 0.92 and m["shadow"] < 0.08:
            s -= 0.12

    return s


def step(state, v_current, history):
    best_state = None
    best_class = None
    best_metrics = None
    best_score = -999.0

    for cls, w in CLASSES.items():
        candidate_state = mutate(state, w)
        metrics = compute_metrics(candidate_state)
        sc = score(cls, metrics, v_current, history)

        if sc > best_score:
            best_score = sc
            best_class = cls
            best_metrics = metrics
            best_state = candidate_state

    return best_class, best_metrics, best_state, round(best_score, 3)

Writing runtime/agent_loop.py


In [ ]:
%%writefile runtime/market_engine.py
def get_phase(step):
    return "DAY" if (step % 40) < 20 else "NIGHT"


def update_vitality(cls, decision, v, phase):
    cost_map = {
        "COMMIT": 0.05,
        "SOFT_COMMIT": 0.025,
        "REJECT": 0.01,
    }
    cost = cost_map.get(decision, 0.02)

    if cls == "TANK":
        cost *= 0.75
    elif cls == "ARCHER":
        cost *= 1.00
    elif cls == "MAGE":
        cost *= 1.20
    elif cls == "HEALER":
        cost *= 0.85
    elif cls == "ASSASSIN":
        cost *= 1.65

    if phase == "NIGHT":
        if cls == "MAGE":
            cost *= 1.15
        if cls == "TANK":
            cost *= 0.92
    else:
        if cls == "ARCHER":
            cost *= 0.95

    bonus = 0.0
    if cls == "HEALER" and decision != "REJECT":
        bonus += 0.08
    elif cls == "MAGE" and decision == "COMMIT":
        bonus += 0.04
    elif cls == "TANK" and decision != "REJECT":
        bonus += 0.02
    elif cls == "ARCHER" and decision == "COMMIT":
        bonus += 0.015

    new_v = v - cost + bonus
    return max(0.05, min(1.0, new_v))

Writing runtime/market_engine.py


In [ ]:
%%writefile runtime/market_transaction.py
def role_transaction(cls, metrics, vitality, history_len):
    shadow = metrics["shadow"]
    coherence = metrics["coherence"]
    target_fit = metrics["target_fit"]

    decision_override = None
    energy_delta = 0.0
    role_score = 0.0

    if cls == "HEALER":
        if vitality < 0.30:
            decision_override = "SOFT_COMMIT"
            energy_delta += 0.08
            role_score += 1.2
        if shadow > 0.12:
            energy_delta += 0.04
            role_score += 0.5

    elif cls == "TANK":
        if shadow > 0.14 or coherence < 0.82:
            decision_override = "SOFT_COMMIT"
            energy_delta += 0.03
            role_score += 0.8
        else:
            energy_delta += 0.01

    elif cls == "MAGE":
        if vitality > 0.55 and coherence > 0.80:
            role_score += 1.0
            energy_delta += 0.02
        else:
            role_score -= 0.4

    elif cls == "ARCHER":
        if 0.30 <= vitality <= 0.75:
            role_score += 0.7
        if target_fit > 0.32:
            role_score += 0.4
            energy_delta += 0.015

    elif cls == "ASSASSIN":
        if shadow > 0.18 or history_len > 12:
            role_score += 0.5
            energy_delta -= 0.03
        else:
            role_score -= 1.0
            energy_delta -= 0.05

    return {
        "decision_override": decision_override,
        "energy_delta": round(energy_delta, 3),
        "role_score": round(role_score, 3),
    }

Writing runtime/market_transaction.py


In [ ]:
import sys
import os

cwd = os.getcwd()
if cwd not in sys.path:
    sys.path.append(cwd)

for k in list(sys.modules.keys()):
    if k.startswith(("core.", "runtime.")):
        del sys.modules[k]

print("Path fixed + cache cleared")
print("cwd =", cwd)
print("root files =", os.listdir())
print("core files =", os.listdir("core"))
print("runtime files =", os.listdir("runtime"))

Path fixed + cache cleared
cwd = /content
root files = ['.config', 'core', 'runtime', 'sample_data']
core files = ['evaluation.py', 'state.py']
runtime files = ['market_transaction.py', 'market_engine.py', 'agent_loop.py']


In [ ]:
from core.state import default_state
from runtime.agent_loop import step
from runtime.market_engine import update_vitality, get_phase
from runtime.market_transaction import role_transaction

print("Imports OK")

Imports OK


In [ ]:
from collections import Counter

state = default_state()
vitality = 0.40
history = []
logs = []

print(f"{'step':<4} {'class':<10} {'decision':<12} {'v':<6} {'phase':<6} {'econ':<6}")
print("-" * 56)

for i in range(100):
    phase = get_phase(i)

    cls, metrics, state, sc = step(state, vitality, history)

    if metrics["coherence"] > 0.87 and metrics["shadow"] <= 0.14:
        decision = "COMMIT"
    elif metrics["coherence"] > 0.74:
        decision = "SOFT_COMMIT"
    else:
        decision = "REJECT"

    tx = role_transaction(
        cls=cls,
        metrics=metrics,
        vitality=vitality,
        history_len=len(history),
    )

    if tx["decision_override"] is not None:
        decision = tx["decision_override"]

    vitality = update_vitality(cls, decision, vitality, phase)
    vitality = max(0.05, min(1.0, vitality + tx["energy_delta"]))

    history.append(cls)

    logs.append({
        "step": i,
        "class": cls,
        "decision": decision,
        "vitality": round(vitality, 3),
        "phase": phase,
        "econ": tx,
        "score": sc,
        "metrics": metrics,
    })

    if i % 5 == 0:
        print(
            f"{i:<4} {cls:<10} {decision:<12} {round(vitality,3):<6} {phase:<6} {tx['role_score']:<6}"
        )

print("-" * 56)
print("done")

step class      decision     v      phase  econ  
--------------------------------------------------------
0    ARCHER     SOFT_COMMIT  0.391  DAY    1.1   
5    MAGE       COMMIT       0.328  DAY    -0.4  
10   HEALER     SOFT_COMMIT  0.431  DAY    1.2   
15   ASSASSIN   SOFT_COMMIT  0.345  DAY    0.5   
20   MAGE       COMMIT       0.404  NIGHT  -0.4  
25   MAGE       SOFT_COMMIT  0.355  NIGHT  -0.4  
30   TANK       SOFT_COMMIT  0.315  NIGHT  0.0   
35   TANK       SOFT_COMMIT  0.467  NIGHT  0.8   
40   ASSASSIN   COMMIT       0.193  DAY    0.5   
45   ARCHER     COMMIT       0.286  DAY    1.1   
50   ARCHER     COMMIT       0.371  DAY    1.1   
55   TANK       COMMIT       0.383  DAY    0.0   
60   MAGE       COMMIT       0.319  NIGHT  -0.4  
65   MAGE       SOFT_COMMIT  0.413  NIGHT  -0.4  
70   TANK       COMMIT       0.301  NIGHT  0.0   
75   ARCHER     SOFT_COMMIT  0.371  NIGHT  1.1   
80   MAGE       SOFT_COMMIT  0.282  DAY    -0.4  
85   TANK       COMMIT       0.396  DAY    

In [ ]:
from collections import Counter

class_counts = Counter([x["class"] for x in logs])
decision_counts = Counter([x["decision"] for x in logs])

print("Evolution:", dict(class_counts))
print("Decisions:", dict(decision_counts))
print("Final vitality:", round(logs[-1]["vitality"], 3))

Evolution: {'ARCHER': 38, 'MAGE': 25, 'TANK': 18, 'HEALER': 12, 'ASSASSIN': 7}
Decisions: {'SOFT_COMMIT': 52, 'COMMIT': 44, 'REJECT': 4}
Final vitality: 0.436
